<div style="background-color: #FFFF00; color: #000000; padding: 12px 18px; font-family: Arial, Helvetica, sans-serif;">
<strong style="color: #000000;">MINE-4101 Ciencia de Datos Aplicada · Universidad de los Andes · 2026-20</strong><br>
<span style="color: #000000;">Proyecto final · Primera entrega · Actividad 6</span>
</div>

# Entendimiento de los datos

Análisis exploratorio y calidad de los datos del levantamiento catastral de un municipio colombiano.

**Integrantes**

- Daniel Felipe Triviño Santana
- Nicolas Bedoya Figueroa
- Joubert Daniel Álvarez
- Gabriel Bejarano

**Datos:** la base catastral de un municipio colombiano, en el modelo LADM-COL y en su versión podada: 45 tablas, de las cuales 20 tienen los datos del levantamiento (410 171 filas) y 23 son dominios. Ver la [descripción de la base](../../datos/base-podada/README.md) y su [diccionario de datos](../../datos/base-podada/diccionario-de-datos.md).

**Qué pide el enunciado** (actividad 6, 35 % de la primera entrega): un reporte de análisis exploratorio y de calidad de los datos, en el que sea evidente el uso de técnicas univariadas y multivariadas, gráficas y no gráficas. El objetivo de la entrega es validar la calidad y la suficiencia de los datos para la solución planteada.

**Estado:** andamiaje. Están la estructura y la carga de librerías; las etapas 2 a 8 están por desarrollar. Los resultados más importantes pasan a la sección 6 del [reporte](../reporte/reporte.pdf).

Última actualización: octubre de 2026.

## Tabla de contenido

Las etapas siguen el proceso de exploración que presenta el curso (Ghosh et al., 2018): cargar los datos, distinguir los atributos, analizarlos uno por uno, buscar interacciones entre ellos, detectar valores aberrantes y faltantes, detectar valores atípicos y recoger los insights. El paso de ingeniería de características de ese proceso corresponde a la preparación de datos de la entrega final (actividad 8) y no se desarrolla aquí.

Los cuatro tipos de análisis que pide el enunciado (univariado y multivariado, gráfico y no gráfico) son las subsecciones de las etapas 4 y 5.

| | Etapa | Qué se hace | Apoyo en el curso |
| --- | --- | --- | --- |
| 1 | [Librerías y configuración](#librerias) | Cargar las librerías y fijar las opciones de tablas y gráficas | Tutoriales 1 y 2 |
| 2 | [Carga de los datos](#carga) | Leer las tablas y revisar dimensiones, tipos y una muestra | Inicio del proceso de exploración; tutoriales 1 y 2 |
| 3 | [Identificación de los atributos](#atributos) | Clasificar cada atributo como cuantitativo o cualitativo | Paso 1; EDA, diapositivas 10 a 16 |
| 4 | [Análisis univariado](#univariado) | Describir cada atributo por separado: 4.1 no gráfico, 4.2 gráfico | Paso 2; EDA, diapositivas 17 a 28 y 57 a 60; tutorial 1 |
| 5 | [Análisis multivariado](#multivariado) | Buscar relaciones entre atributos: 5.1 no gráfico, 5.2 gráfico | Paso 3; EDA, diapositivas 29 a 47 y 61 a 70; tutorial 1 |
| 6 | [Calidad de los datos](#calidad) | 6.1 valores faltantes y aberrantes, 6.2 valores atípicos, 6.3 resumen de problemas | Pasos 4 y 5; Calidad, diapositivas 4 a 22; tutorial 2 |
| 7 | [Suficiencia de los datos](#suficiencia) | Concluir si los datos alcanzan para la solución planteada | Objetivos de la primera entrega, en el enunciado |
| 8 | [Conclusiones e insights](#conclusiones) | Reunir los hallazgos de las etapas anteriores | Cierre del proceso de exploración; EDA, diapositivas 4 a 6 |

Fuentes del curso:

- **EDA:** diapositivas «Análisis exploratorio de datos (EDA)», clases 3 y 4.
- **Calidad:** diapositivas «Calidad y limpieza de datos», clase 5.
- **Tutoriales 1 y 2:** [data-analysis-tutorial](https://github.com/datascience-uniandes/data-analysis-tutorial) y [data-quality-tutorial](https://github.com/datascience-uniandes/data-quality-tutorial).
- **Proceso de exploración:** A. Ghosh et al., "A comprehensive review of tools for exploratory analysis of tabular industrial datasets", *Visual Informatics* 2(4), 2018. Es el que muestra la diapositiva 9 de EDA.

<a id="librerias"></a>

## 1. Librerías y configuración

Las librerías son las que usan los tutoriales del curso y están en el entorno del proyecto (ver el [README](../../README.md#entorno-de-python)). Las que solo necesita una etapa (`scipy`, `statsmodels`, `scikit-learn`, `ydata-profiling`, `Levenshtein`) se agregan a esta primera celda cuando esa etapa se desarrolle.

In [1]:
# Librerías básicas de análisis y visualización
import platform
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import font_manager

In [2]:
# Raíz del repositorio: la carpeta que contiene pyproject.toml
RAIZ = next(c for c in [Path.cwd(), *Path.cwd().parents] if (c / "pyproject.toml").exists())

# Tablas: todas las columnas, hasta 100 filas y sin notación científica
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.options.display.float_format = "{:.2f}".format

# Gráficas: identidad visual del proyecto (ver identidad/README.md)
plt.style.use(RAIZ / "identidad" / "graficas.mplstyle")
AMARILLO, NEGRO, GRIS = "#FFFF00", "#000000", "#666666"

# Semilla para todo lo que use azar (muestras, reducción de dimensionalidad)
SEMILLA = 4101

In [3]:
# Versiones con las que se ejecutó el notebook
print(f"Python {platform.python_version()}")
for libreria in (np, pd, mpl, sns):
    print(f"{libreria.__name__} {libreria.__version__}")

# Fuente que usan las gráficas: debería ser Arial (ver identidad/README.md)
ruta_fuente = font_manager.findfont(font_manager.FontProperties(family=plt.rcParams["font.family"]))
print("Fuente de las gráficas:", font_manager.get_font(ruta_fuente).family_name)

Python 3.12.13
numpy 2.3.5
pandas 2.3.3
matplotlib 3.10.0
seaborn 0.13.2
Fuente de las gráficas: Arial


<a id="carga"></a>

## 2. Carga de los datos

Como en los tutoriales del curso: leer cada tabla y revisar sus dimensiones, los tipos de sus columnas y una muestra de filas.

**Antes de desarrollarla hay dos decisiones pendientes** (ver los pendientes de [datos/](../../datos/README.md#pendientes) y de la [base podada](../../datos/base-podada/README.md#pendientes)):

- Cómo lee el notebook la base. Hoy está en una instalación local de PostgreSQL y el entorno del proyecto no tiene una librería para conectarse a ella.
- Con qué datos lo ejecuta quien clone el repositorio. Los notebooks deben poder ejecutarse sin errores, y la base no es pública.

**Lo que las salidas no pueden mostrar.** El repositorio es público y las salidas de las celdas se suben con el notebook. Ninguna debe dejar a la vista (ver [Antes de compartirla](../../datos/base-podada/README.md#antes-de-compartirla)):

- Datos personales: los documentos de identidad, nombres, teléfonos, correos y domicilios de `cr_interesado` y `lc_contactovisita`.
- Lo que identifica al municipio: `departamento` y `municipio` de `lc_predio`, los primeros dígitos de `numero_predial`, el nombre del archivo de origen en `t_ili2db_dataset` y las coordenadas de las geometrías.

In [4]:
# Pendiente

<a id="atributos"></a>

## 3. Identificación de los atributos

Primer paso del proceso de exploración: distinguir de qué tipo es cada atributo, porque de eso depende qué medidas y qué gráficas le sirven.

| Tipo | Subtipos | Cómo reconocerlo |
| --- | --- | --- |
| Cuantitativo | Discreto o continuo | Representa una magnitud: restar dos valores tiene sentido. Es continuo si además tiene sentido el valor medio entre dos valores |
| Cualitativo (categórico) | Nominal u ordinal | Toma niveles. Es ordinal si los niveles se pueden ordenar |

En esta base, el tipo de dato de la columna no basta para clasificarla:

- Los atributos de dominio (`condicion_predio`, `uso`, `tipo`) guardan el `t_id` de una fila de la tabla del dominio. Son cualitativos, aunque en la base sean números.
- Las llaves y los identificadores (`t_id`, `numero_predial`, `matricula_inmobiliaria`) tampoco son cuantitativos: restar dos de ellos no significa nada.

In [5]:
# Pendiente

<a id="univariado"></a>

## 4. Análisis univariado

Describe cada atributo por separado.

El tutorial 1 incluye un perfilamiento automático (`ydata-profiling`), que sirve como primera mirada a todas las columnas de una tabla. Su reporte en HTML incluye muestras de los datos: no se sube al repositorio.

### 4.1 No gráfico

Medidas estadísticas que resumen la distribución de un atributo:

- **Tabla de frecuencias:** valores únicos, valor más común (moda), frecuencia absoluta y relativa.
- **Tendencia central:** media y mediana. La mediana es robusta a los valores extremos.
- **Dispersión:** varianza y desviación estándar; la desviación absoluta mediana (MAD) como alternativa robusta.
- **Posición:** cuantiles, rango intercuartílico, mínimo y máximo.
- **Forma:** asimetría y curtosis.

In [6]:
# Pendiente

### 4.2 Gráfico

- **Barras** para las frecuencias de un atributo cualitativo.
- **Histograma** para la distribución de uno cuantitativo. Hay que elegir el número de intervalos.
- **Diagrama de caja** para la tendencia central, la asimetría y los valores atípicos. No muestra bien las distribuciones con varias modas.

In [7]:
# Pendiente

<a id="multivariado"></a>

## 5. Análisis multivariado

Describe la distribución conjunta de varios atributos o la relación entre ellos. En la práctica casi siempre se hace de dos en dos.

### 5.1 No gráfico

- **Tablas cruzadas** entre atributos cualitativos, o cuantitativos con pocos valores.
- **Correlación** entre atributos cuantitativos: lineal (Pearson) y monótona (Spearman), y la matriz de correlación cuando son varios. Correlación no implica causalidad.
- **Regresión lineal**, para estimar un atributo a partir de otros y leer sus coeficientes. Tampoco implica causalidad y tiene supuestos que deben cumplirse.
- **Análisis de componentes principales (PCA)**, para resumir muchos atributos cuantitativos en pocos componentes.
- **Pruebas de hipótesis** (t, ANOVA, chi-cuadrado; clase 6 y tutorial 3), cuando haga falta confirmar una diferencia o una asociación.

In [8]:
# Pendiente

### 5.2 Gráfico

- **Diagrama de dispersión** entre dos atributos cuantitativos, con el color o la forma de los puntos para una categoría.
- **Mapa de calor** para una magnitud según dos atributos cualitativos, o para la matriz de correlación.
- **Gráficas por categoría:** la distribución de un atributo cuantitativo en cada nivel de uno cualitativo.
- **Reducción de dimensionalidad** para ver muchos atributos en dos dimensiones. PCA y t-SNE están en `scikit-learn`; UMAP necesitaría una dependencia nueva.
- **Mapas**, solo si la posición geográfica es relevante. Aquí, además, un mapa del municipio lo identifica (ver la [etapa 2](#carga)).

In [9]:
# Pendiente

<a id="calidad"></a>

## 6. Calidad de los datos

El curso define nueve dimensiones de calidad: completitud, consistencia, conformidad, relevancia, precisión, procesabilidad, disponibilidad, temporalidad y credibilidad. Y clasifica los problemas según el nivel en el que aparecen:

| Nivel | Problemas |
| --- | --- |
| Atributo | Valor faltante, violación de sintaxis, error de digitación, violación de dominio |
| Registro | Violación de integridad: valores de un mismo registro que se contradicen |
| Columna | Sinónimos, homónimos, violación de singularidad |
| Tabla | Duplicados aproximados |
| Múltiples tablas | Representación heterogénea de los datos, sinónimos, granularidades distintas |

Aquí los problemas se diagnostican. La limpieza corresponde a la preparación de datos de la entrega final (actividad 8).

### 6.1 Valores faltantes y aberrantes

Paso 4 del proceso de exploración. El [diccionario de datos](../../datos/base-podada/diccionario-de-datos.md) ya trae, para cada columna, el porcentaje de filas que tienen dato; es el punto de partida para la completitud. Técnicas del tutorial 2 para lo demás:

- Diccionarios de valores válidos. Aquí ese papel lo cumplen las tablas de dominio.
- Expresiones regulares para comprobar formatos.
- Duplicados, por un conjunto de atributos o por el registro entero.
- Medidas de similitud entre textos (distancia de Levenshtein) para los errores de digitación.
- Consistencia entre columnas del mismo registro.

In [10]:
# Pendiente

### 6.2 Valores atípicos

Paso 5 del proceso de exploración. Las dos definiciones estándar del curso: valores a más de 3 desviaciones estándar de la media, o a más de 1,5 veces el rango intercuartílico por debajo del primer cuartil o por encima del tercero. Qué cuenta como atípico depende muchas veces del negocio.

In [11]:
# Pendiente

### 6.3 Resumen de problemas de calidad

Como pide la conclusión del tutorial 2: cada problema encontrado, la dimensión de calidad que afecta y la estrategia para resolverlo o mitigarlo.

| Problema | Tabla y atributo | Dimensión | Nivel | Estrategia |
| --- | --- | --- | --- | --- |
| Pendiente | | | | |

<a id="suficiencia"></a>

## 7. Suficiencia de los datos

Responde, con lo encontrado en las etapas anteriores, si los datos alcanzan para la solución planteada: si están los atributos que necesitan las preguntas de negocio, cuántos registros utilizables quedan y qué haría falta conseguir.

Pendiente. Depende de las preguntas de negocio del enfoque analítico (actividad 4).

<a id="conclusiones"></a>

## 8. Conclusiones e insights

Los hallazgos de las etapas anteriores, escritos como insights: un entendimiento profundo, no obvio, significativo (con valor potencial), accionable (abre nuevas preguntas) y basado en los datos. Alimentan la sección 6 del reporte y las conclusiones iniciales (actividad 7).

Pendiente.